In [7]:
import os
import sys

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_gigachat.chat_models import GigaChat
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

load_dotenv()

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

credentials = os.getenv("GIGACHAT_CREDENTIALS") or os.getenv("GIGA_KEY")

llm = GigaChat(
    credentials=credentials,
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.2,
    max_tokens=1000,
)

import pandas as pd

response = llm.invoke("Привет! Как дела?")
print(response.content)

df = pd.read_csv("rental_32.csv", sep=";").head(15)

test_texts = {
    f"request_{idx + 1}": text
    for idx, text in enumerate(df["text"].fillna(""))
}

expected_amounts = {
    f"request_{idx + 1}": int(amount)
    for idx, amount in enumerate(df["amount"])
}

import re

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Извлеки количество проживающих из текста заявки на аренду. "
            "Верни только одно целое число без слов и пояснений."
        ),
        (
            "human",
            "Текст заявки:\n{request_text}",
        ),
    ]
)

chain = prompt | llm | StrOutputParser()

predicted_amounts = {}
for key, text in test_texts.items():
    raw = chain.invoke({"request_text": text}).strip()
    match = re.search(r"\d+", raw)
    predicted_amounts[key] = int(match.group(0)) if match else -1

results = pd.DataFrame(
    {
        "request_id": list(test_texts.keys()),
        "expected_amount": [expected_amounts[k] for k in test_texts],
        "predicted_amount": [predicted_amounts[k] for k in test_texts],
        "text": [test_texts[k] for k in test_texts],
    }
)
results["is_correct"] = results["expected_amount"] == results["predicted_amount"]

print(results[["request_id", "expected_amount", "predicted_amount", "is_correct"]])
print(f"accuracy: {results['is_correct'].mean():.2%}")


Привет! Всё отлично, готов общаться и помогать тебе с любыми вопросами. А ты как?
    amount  predicted_amount  is_correct  \
0        4                 5       False   
1        4                 4        True   
2        6                 6        True   
3        2                 2        True   
4        4                 2       False   
5        2                 2        True   
6        2                 2        True   
7        3                 3        True   
8        3                 3        True   
9        3                 3        True   
10       3                 3        True   
11       4                 8       False   
12       5                 5        True   
13       1                 0       False   
14       2                 2        True   

                                                 text  
0   Снимем жилье с 30.06-7.07 Два взрослых и 2дете...  
1   Добрый день! Ищем жильё (гостевой дом) с парко...  
2   Добрый вечер! ищем жилье на отдых с 16-29